# Experiment 4: Logistic Regression Hyperparameter Tuning with MLflow

Grid search optimization for Logistic Regression hyperparameters (solvers, penalties, C regularization, l1_ratio) on TF-IDF features with full MLflow & DagsHub experiment tracking.

In [ ]:
import os
import re
import string
import numpy as np
import pandas as pd
import warnings

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

import mlflow
import mlflow.sklearn
import dagshub

warnings.simplefilter("ignore", UserWarning)
warnings.filterwarnings("ignore")

## 1. Setup MLflow Tracking

In [ ]:
DATA_PATH = '../data/balanced_sentiment_dataset.csv' if os.path.exists('../data/balanced_sentiment_dataset.csv') else 'data/balanced_sentiment_dataset.csv'

MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "https://dagshub.com/rohitkr8527/sentiment-analysis.mlflow")
REPO_OWNER = os.getenv("DAGSHUB_REPO_OWNER", "rohitkr8527")
REPO_NAME = os.getenv("DAGSHUB_REPO_NAME", "sentiment-analysis")

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
dagshub_token = os.getenv("DAGSHUB_TOKEN")
if dagshub_token:
    os.environ["MLFLOW_TRACKING_USERNAME"] = dagshub_token
    os.environ["MLFLOW_TRACKING_PASSWORD"] = dagshub_token
    dagshub.init(repo_owner=REPO_OWNER, repo_name=REPO_NAME, mlflow=True)

mlflow.set_experiment("LR Hyperparameter Tuning")
print("MLflow experiment configured: 'LR Hyperparameter Tuning'")

## 2. Load & Prepare TF-IDF Features

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(f"[{re.escape(string.punctuation)}]", " ", text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    words = [lemmatizer.lemmatize(w) for w in text.split() if w not in stop_words]
    return " ".join(words)

df = pd.read_csv(DATA_PATH)
df["text"] = df["text"].astype(str).apply(preprocess_text)

vectorizer = TfidfVectorizer(max_features=10000)
X = vectorizer.fit_transform(df["text"])
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training features: {X_train.shape}, Test features: {X_test.shape}")

## 3. Run GridSearchCV and Log All Trials to MLflow

In [ ]:
param_grid = [
    {
        "solver": ["liblinear"],
        "penalty": ["l1", "l2"],
        "C": [0.1, 1.0, 10.0]
    },
    {
        "solver": ["saga"],
        "penalty": ["elasticnet"],
        "C": [0.1, 1.0, 10.0],
        "l1_ratio": [0.0, 0.5, 1.0]
    },
    {
        "solver": ["lbfgs"],
        "penalty": ["l2"],
        "C": [0.1, 1.0, 10.0]
    }
]

with mlflow.start_run(run_name="GridSearchCV_LogisticRegression_Tuning") as parent_run:
    grid_search = GridSearchCV(
        LogisticRegression(max_iter=500),
        param_grid=param_grid,
        cv=3,
        scoring="f1",
        n_jobs=-1,
        verbose=1
    )
    grid_search.fit(X_train, y_train)

    # Log each hyperparameter combination as a nested child run
    for params, mean_score, std_score in zip(
        grid_search.cv_results_["params"],
        grid_search.cv_results_["mean_test_score"],
        grid_search.cv_results_["std_test_score"]
    ):
        with mlflow.start_run(run_name=f"LR_params_{params}", nested=True):
            model = LogisticRegression(**params, max_iter=500, random_state=42)
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

            metrics = {
                "accuracy": float(accuracy_score(y_test, y_pred)),
                "precision": float(precision_score(y_test, y_pred)),
                "recall": float(recall_score(y_test, y_pred)),
                "f1_score": float(f1_score(y_test, y_pred)),
                "mean_cv_f1": float(mean_score),
                "std_cv_f1": float(std_score)
            }

            mlflow.log_params(params)
            mlflow.log_metrics(metrics)

    # Log the best model artifact to parent run
    best_params = grid_search.best_params_
    best_model = grid_search.best_estimator_
    best_f1 = float(grid_search.best_score_)

    mlflow.log_params(best_params)
    mlflow.log_metric("best_cv_f1_score", best_f1)
    mlflow.sklearn.log_model(best_model, "best_model")

    print(f"\n--- Best Hyperparameters: {best_params} ---")
    print(f"Best CV F1-Score: {best_f1:.4f}")